# Meshes · One face, connectivity and edge vectors
**CAS Computational Form-finding · Block Research Group · ETH Zürich**

**Why this is needed:** a graph records connections; a mesh also records which
ordered vertex cycles bound surfaces. That extra information is needed for face
areas, normals and surface neighbourhoods. Start with **one quadrilateral face**.

Coordinates are in metres, area is in m², and connectivity matrix entries are
dimensionless. **Predict:** how many boundary edges does a single quad have?


## 1. Start Python
Run all cells in order with **Run → Run All Cells**, or step through with **Shift+Enter**.
JupyterLite supplies NumPy and SciPy. On desktop Jupyter, install them with
`%pip install -q numpy scipy` if an import below reports a missing package.
The pinned COMPAS packages below are the course's tested versions; `--no-deps`
avoids installing a desktop notebook server in browser Python.


In [ ]:
%pip install -q compas==2.15.1 pythreejs==2.4.2
%pip install -q compas_notebook==0.11.0 --no-deps


In [ ]:
# Enable widgets on Colab; browser JupyterLite and desktop Jupyter need no change.
try:
    from google.colab import output
except ImportError:
    pass
else:
    output.enable_custom_widget_manager()


In [ ]:
import numpy as np
from compas.datastructures import Mesh
from compas.matrices import connectivity_matrix
from compas_notebook import Viewer
from compas_notebook.config import Config

width, depth = 2.0, 1.0  # m
mesh = Mesh()
coordinates = {10: [0, 0, 0], 20: [width, 0, 0],
               30: [width, depth, 0], 40: [0, depth, 0]}
for key, xyz in coordinates.items():
    mesh.add_vertex(key=key, x=xyz[0], y=xyz[1], z=xyz[2])
face = mesh.add_face([10, 20, 30, 40])
print(mesh)
print("Face cycle:", mesh.face_vertices(face))
print("Vertex 10 neighbours:", mesh.vertex_neighbors(10))
print("Face normal:", mesh.face_normal(face))
assert mesh.number_of_vertices() == 4 and mesh.number_of_edges() == 4
assert mesh.number_of_faces() == 1
assert np.isclose(mesh.area(), width * depth)
assert mesh.face_normal(face)[2] > 0
assert all(mesh.is_vertex_on_boundary(key) for key in mesh.vertices())


## From mesh keys to array rows
Vertex keys `10, 20, 30, 40` are identifiers, not array row indices. Save the map
before constructing arrays. Each row of oriented incidence `C` has **−1 at an
edge's start** and **+1 at its end**. Consequently `C @ xyz` gives end-minus-start
edge vectors. Their directions are bookkeeping and do not prescribe a structural force.

We pass explicitly indexed edge pairs to `compas.matrices.connectivity_matrix`, keeping matrix rows in the printed edge order.


In [ ]:
vertex_keys = list(mesh.vertices())
key_to_row = mesh.vertex_index()
edges = list(mesh.edges())
indexed_edges = [(key_to_row[u], key_to_row[v]) for u, v in edges]
C = np.asarray(connectivity_matrix(indexed_edges, rtype="array"))
xyz = np.asarray([mesh.vertex_coordinates(key) for key in vertex_keys])
edge_vectors = C @ xyz
print("Vertex key → row:", key_to_row)
print("Edge row → oriented edge:", dict(enumerate(edges)))
print("Incidence C:\n", C)
print("Edge vectors [m]:\n", edge_vectors)
assert C.shape == (len(edges), len(vertex_keys))
assert np.allclose(C @ np.ones(len(vertex_keys)), 0)
for row, (u, v) in enumerate(edges):
    assert C[row, key_to_row[u]] == -1 and C[row, key_to_row[v]] == 1
    assert np.allclose(edge_vectors[row], np.array(coordinates[v]) - coordinates[u])

config = Config()
config.view.width = 700
config.view.height = 360
config.view.show_grid = False

config.view.camera.position = [4, -5, 4]
config.view.camera.target = [width / 2, depth / 2, 0]
view = Viewer(config=config)
view.scene.add(mesh, color="#b6cae7", show_edges=True, show_vertices=True)
view.show()


## Predict → change → check
Starting result: **4 vertices, 4 edges, 1 face**, area **2 m²**; `C` is **4 × 4**.

1. Change width to 3 m. Predict which matrix entries and geometric quantities change.
2. Explain why vertex key `40` uses row `3`, rather than row `40`.
3. Reverse the face cycle. Predict how the normal changes.
4. Sketch how two triangles could replace this quad, and count the extra edge.

The final cell constructs the two-triangle alternative with the same points.


In [ ]:
# Safe experiment: a new mesh with two faces and one extra diagonal.
trial = Mesh()
for key, xyz in coordinates.items():
    trial.add_vertex(key=key, x=xyz[0], y=xyz[1], z=xyz[2])
trial.add_face([10, 20, 30])
trial.add_face([10, 30, 40])
assert trial.number_of_faces() == 2 and trial.number_of_edges() == 5
assert np.isclose(trial.area(), mesh.area())
print("Two triangles:", trial, "; unchanged area:", trial.area(), "m²")


## Sources
Original minimal exercise informed by [Seoul 2025 mesh examples](https://github.com/BRG-teaching/seoul_2025/tree/b641b732a3e337eb3a19d7a91048946820695e7a/05_compas)
and [ITA21 topology](https://github.com/compas-teaching/ITA21/tree/7213b322067efe6102e998e83194e8dbe39a82bf). API: [COMPAS Mesh](https://compas.dev/compas/2.15.1/api/compas.datastructures.html).
Viewer pattern: [COMPAS JupyterLite](https://github.com/compas-dev/compas-jupyterlite/tree/d9df5aef10685c80507236554900d72b86504d6a).

Save with **Ctrl+S** (or **⌘S**), then use **File → Save and Export Notebook As → Notebook** to download your work. Browser files are local to this browser; export a copy before moving machines.
